# DEAI-opdrachten Classificatie – Ames Housing
**Binaire classificatie:** voorspellen of een huis wel/geen garage heeft
**Multi-class classificatie:** voorspellen van het kwaliteitsniveau (Overall Qual)

In [28]:
import pandas as pd
import numpy as np

from IPython.display import display
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [29]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [30]:
bestand = "AmesHousing.xlsx"

df = pd.read_excel(bestand, sheet_name="AmesHousing")
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")

print("Vorm van de dataset:", df.shape)
display(df.head())
display(data_dictionary)

Vorm van de dataset: (2930, 12)


,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


,Variabele,Betekenis
0,ID,"Uniek nummer per huis, te vergelijken met een ..."
1,SalePrice,Verkoopprijs van het huis (in dollars: USD)
2,Garage,Geeft weer of het huis wel/geen garage bevat
3,Overall Qual,Algemene kwaliteit van materialen en afwerking...
4,Gr Liv Area,Woonoppervlak boven de grond (square feet)
5,Total Bsmt SF,Totale oppervlakte van de kelder
6,Lot Area,Grootte van het perceel (square feet)
7,Year Built,Bouwjaar van het huis
8,Full Bath,Aantal volledige badkamers
9,Bedroom AbvGr,Aantal slaapkamers boven de grond


In [31]:
# Deze regel kun je gebruiken om alle hyperparameters te bekijken:
# help(DecisionTreeClassifier)

print("Hyperparameters die ik in dit notebook gebruik:")
print("- max_depth")
print("- min_samples_split")
print("- min_samples_leaf")

Hyperparameters die ik in dit notebook gebruik:
- max_depth
- min_samples_split
- min_samples_leaf


In [32]:
# Selecteert features, vult missende waarden op en doet one-hot encoding
def maak_features(dataframe, feature_kolommen):
    X = dataframe[feature_kolommen].copy()

    for kolom in X.columns:
        if is_numeric_dtype(X[kolom]):
            X[kolom] = X[kolom].fillna(X[kolom].median())
        else:
            modus = X[kolom].mode(dropna=True)
            if len(modus) > 0:
                X[kolom] = X[kolom].fillna(modus.iloc[0])
            else:
                X[kolom] = X[kolom].fillna("Onbekend")

    categorische_kolommen = X.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()

    X = pd.get_dummies(X, columns=categorische_kolommen, drop_first=False)

    return X

In [33]:
def run_experiment(dataframe, target_kolom, feature_kolommen, hyperparameters):
    """Voert één experiment uit en geeft alle belangrijke resultaten terug."""

    data = dataframe[feature_kolommen + [target_kolom]].copy()
    data = data.dropna(subset=[target_kolom])

    X = maak_features(data, feature_kolommen)
    y = data[target_kolom].copy()

    # Horizontale en verticale split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    model = DecisionTreeClassifier(random_state=42, **hyperparameters)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    labels = sorted(y.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    resultaat = {
        "features": feature_kolommen,
        "hyperparameters": hyperparameters,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "confusion_matrix": cm,
        "labels": labels,
        "report": classification_report(y_test, y_pred, zero_division=0),
        "X_train_shape": X_train.shape,
        "X_test_shape": X_test.shape,
        "y_train_shape": y_train.shape,
        "y_test_shape": y_test.shape
    }

    return resultaat

In [34]:
def maak_resultaten_tabel(resultaten_dict):
    rijen = []

    for naam, res in resultaten_dict.items():
        rijen.append({
            "Run": naam,
            "Features": ", ".join(res["features"]),
            "Hyperparameters": str(res["hyperparameters"]),
            "Accuracy": round(res["accuracy"], 4),
            "Precision macro": round(res["precision_macro"], 4),
            "Recall macro": round(res["recall_macro"], 4),
            "F1 macro": round(res["f1_macro"], 4)
        })

    return pd.DataFrame(rijen)

In [35]:
garage_target = "Garage"
garage_features_eerste_run = ["SalePrice", "Gr Liv Area", "Neighborhood"]

garage_data = df[garage_features_eerste_run + [garage_target]].dropna(subset=[garage_target]).copy()

X_garage = maak_features(garage_data, garage_features_eerste_run)
y_garage = garage_data[garage_target].copy()

print("Horizontale split:")
print("X_garage =", X_garage.shape)
print("y_garage =", y_garage.shape)

X_train_garage, X_test_garage, y_train_garage, y_test_garage = train_test_split(
    X_garage, y_garage, test_size=0.20, random_state=42, stratify=y_garage
)

print("\nVerticale split:")
print("X_train_garage =", X_train_garage.shape)
print("X_test_garage  =", X_test_garage.shape)
print("y_train_garage =", y_train_garage.shape)
print("y_test_garage  =", y_test_garage.shape)

Horizontale split:
X_garage = (2930, 30)
y_garage = (2930,)

Verticale split:
X_train_garage = (2344, 30)
X_test_garage  = (586, 30)
y_train_garage = (2344,)
y_test_garage  = (586,)


In [36]:
garage_experimenten = {
    "Initial run": {
        "features": ["SalePrice", "Gr Liv Area", "Neighborhood"],
        "params": {"max_depth": 4, "min_samples_split": 10},
        "reden": "Starten met de verwachte top 3 features."
    },
    "Experiment 1": {
        "features": ["SalePrice", "Gr Liv Area", "Neighborhood"],
        "params": {"max_depth": 6, "min_samples_split": 10},
        "reden": "Kijken of een diepere boom betere patronen vindt."
    },
    "Experiment 2": {
        "features": ["SalePrice", "Gr Liv Area", "Year Built", "Neighborhood"],
        "params": {"max_depth": 4, "min_samples_split": 10},
        "reden": "Extra feature Year Built toevoegen."
    },
    "Experiment 3": {
        "features": ["SalePrice", "Gr Liv Area", "Year Built", "Neighborhood"],
        "params": {"max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 2},
        "reden": "Nog een keer tunen met extra feature en extra hyperparameter."
    }
}

garage_resultaten = {}

for naam, info in garage_experimenten.items():
    garage_resultaten[naam] = run_experiment(
        df,
        target_kolom="Garage",
        feature_kolommen=info["features"],
        hyperparameters=info["params"]
    )

garage_tabel = maak_resultaten_tabel(garage_resultaten)
display(garage_tabel)

,Run,Features,Hyperparameters,Accuracy,Precision macro,Recall macro,F1 macro
0,Initial run,"SalePrice, Gr Liv Area, Neighborhood","{'max_depth': 4, 'min_samples_split': 10}",0.9454,0.7235,0.5147,0.5154
1,Experiment 1,"SalePrice, Gr Liv Area, Neighborhood","{'max_depth': 6, 'min_samples_split': 10}",0.9420,0.6408,0.5276,0.5376
2,Experiment 2,"SalePrice, Gr Liv Area, Year Built, Neighborhood","{'max_depth': 4, 'min_samples_split': 10}",0.9471,0.7750,0.5451,0.5674
3,Experiment 3,"SalePrice, Gr Liv Area, Year Built, Neighborhood","{'max_depth': 10, 'min_samples_split': 10, 'mi...",0.9454,0.7314,0.6619,0.6894


In [37]:
beste_garage_run = garage_tabel.sort_values("F1 macro", ascending=False).iloc[0]["Run"]
beste_garage = garage_resultaten[beste_garage_run]

print("Beste garage-run op basis van F1 macro:", beste_garage_run)
print()
print("Classification report:")
print(beste_garage["report"])

garage_cm_df = pd.DataFrame(
    beste_garage["confusion_matrix"],
    index=[f"Werkelijk: {label}" for label in beste_garage["labels"]],
    columns=[f"Voorspeld: {label}" for label in beste_garage["labels"]]
)

display(garage_cm_df)

Beste garage-run op basis van F1 macro: Experiment 3

Classification report:
              precision    recall  f1-score   support

          no       0.50      0.34      0.41        32
         yes       0.96      0.98      0.97       554

    accuracy                           0.95       586
   macro avg       0.73      0.66      0.69       586
weighted avg       0.94      0.95      0.94       586



,Voorspeld: no,Voorspeld: yes
Werkelijk: no,11,21
Werkelijk: yes,11,543


In [38]:
qual_target = "Overall Qual"
qual_features_eerste_run = ["SalePrice", "Year Built", "Neighborhood"]

qual_data = df[qual_features_eerste_run + [qual_target]].dropna(subset=[qual_target]).copy()

X_qual = maak_features(qual_data, qual_features_eerste_run)
y_qual = qual_data[qual_target].copy()

print("Horizontale split:")
print("X_qual =", X_qual.shape)
print("y_qual =", y_qual.shape)

X_train_qual, X_test_qual, y_train_qual, y_test_qual = train_test_split(
    X_qual, y_qual, test_size=0.20, random_state=42, stratify=y_qual
)

print("\nVerticale split:")
print("X_train_qual =", X_train_qual.shape)
print("X_test_qual  =", X_test_qual.shape)
print("y_train_qual =", y_train_qual.shape)
print("y_test_qual  =", y_test_qual.shape)

Horizontale split:
X_qual = (2930, 30)
y_qual = (2930,)

Verticale split:
X_train_qual = (2344, 30)
X_test_qual  = (586, 30)
y_train_qual = (2344,)
y_test_qual  = (586,)


In [39]:
qual_experimenten = {
    "Initial run": {
        "features": ["SalePrice", "Year Built", "Neighborhood"],
        "params": {"max_depth": 4, "min_samples_split": 20},
        "reden": "Starten met de verwachte top 3 features."
    },
    "Experiment 1": {
        "features": ["SalePrice", "Year Built", "Neighborhood"],
        "params": {"max_depth": 7, "min_samples_split": 20, "min_samples_leaf": 5},
        "reden": "Eerst kijken of betere hyperparameters al helpen."
    },
    "Experiment 2": {
        "features": ["SalePrice", "Year Built", "Gr Liv Area", "Neighborhood"],
        "params": {"max_depth": 6, "min_samples_split": 10},
        "reden": "Daarna extra feature Gr Liv Area toevoegen."
    },
    "Experiment 3": {
        "features": ["SalePrice", "Year Built", "Gr Liv Area", "Neighborhood", "House Style"],
        "params": {"max_depth": 6, "min_samples_split": 10},
        "reden": "Nog een categorische feature toevoegen."
    }
}

qual_resultaten = {}

for naam, info in qual_experimenten.items():
    qual_resultaten[naam] = run_experiment(
        df,
        target_kolom="Overall Qual",
        feature_kolommen=info["features"],
        hyperparameters=info["params"]
    )

qual_tabel = maak_resultaten_tabel(qual_resultaten)
display(qual_tabel)

,Run,Features,Hyperparameters,Accuracy,Precision macro,Recall macro,F1 macro
0,Initial run,"SalePrice, Year Built, Neighborhood","{'max_depth': 4, 'min_samples_split': 20}",0.5324,0.3123,0.3190,0.3096
1,Experiment 1,"SalePrice, Year Built, Neighborhood","{'max_depth': 7, 'min_samples_split': 20, 'min...",0.5529,0.4345,0.3730,0.3860
2,Experiment 2,"SalePrice, Year Built, Gr Liv Area, Neighborhood","{'max_depth': 6, 'min_samples_split': 10}",0.5478,0.3294,0.3315,0.3291
3,Experiment 3,"SalePrice, Year Built, Gr Liv Area, Neighborho...","{'max_depth': 6, 'min_samples_split': 10}",0.5529,0.3355,0.3314,0.3311


In [40]:
beste_qual_run = qual_tabel.sort_values("F1 macro", ascending=False).iloc[0]["Run"]
beste_qual = qual_resultaten[beste_qual_run]

print("Beste quality-run op basis van F1 macro:", beste_qual_run)
print()
print("Classification report:")
print(beste_qual["report"])

qual_cm_df = pd.DataFrame(
    beste_qual["confusion_matrix"],
    index=[f"Werkelijk: {label}" for label in beste_qual["labels"]],
    columns=[f"Voorspeld: {label}" for label in beste_qual["labels"]]
)

display(qual_cm_df)

Beste quality-run op basis van F1 macro: Experiment 1

Classification report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.25      0.33      0.29         3
           3       0.50      0.12      0.20         8
           4       0.38      0.33      0.36        45
           5       0.61      0.65      0.63       165
           6       0.49      0.52      0.51       146
           7       0.58      0.58      0.58       121
           8       0.60      0.64      0.62        70
           9       0.73      0.38      0.50        21
          10       0.20      0.17      0.18         6

    accuracy                           0.55       586
   macro avg       0.43      0.37      0.39       586
weighted avg       0.55      0.55      0.55       586



,Voorspeld: 1,Voorspeld: 2,Voorspeld: 3,Voorspeld: 4,Voorspeld: 5,Voorspeld: 6,Voorspeld: 7,Voorspeld: 8,Voorspeld: 9,Voorspeld: 10
Werkelijk: 1,0,1,0,0,0,0,0,0,0,0
Werkelijk: 2,0,1,0,2,0,0,0,0,0,0
Werkelijk: 3,0,1,1,5,1,0,0,0,0,0
Werkelijk: 4,0,1,1,15,21,6,1,0,0,0
Werkelijk: 5,0,0,0,13,107,40,5,0,0,0
Werkelijk: 6,0,0,0,4,42,76,23,1,0,0
Werkelijk: 7,0,0,0,0,4,30,70,16,1,0
Werkelijk: 8,0,0,0,0,0,2,21,45,1,1
Werkelijk: 9,0,0,0,0,0,0,0,10,8,3
Werkelijk: 10,0,0,0,0,0,0,1,3,1,1


In [41]:
print(df[["SalePrice", "Gr Liv Area", "Neighborhood"]].dtypes)

SalePrice       int64
Gr Liv Area     int64
Neighborhood      str
dtype: object


### Korte conclusie model 2
- Dit model is duidelijk moeilijker, omdat er 10 kwaliteitsniveaus voorspeld moeten worden.
- Hier bleek dat hyperparameter tuning eerst meer hielp dan meteen extra features toevoegen.
- `Gr Liv Area` en `House Style` waren wel logisch om te testen, maar gaven niet de beste macro F1-score.

Beste run voor Overall Qual: `Experiment 1`


## Stap 8 – Verantwoording van de experimenten

### Wat inspireerde de volgende experimenten?

#### Garage
1. Ik begon met mijn verwachte top 3 features.
2. Daarna probeerde ik eerst een diepere boom.
3. Omdat dat nog niet genoeg hielp, voegde ik Year Built toe.
4. Toen zag ik verbetering, dus daarna heb ik nog een extra hyperparameter (`min_samples_leaf`) toegevoegd.

#### Overall Qual
1. Ik begon weer met mijn verwachte top 3 features.
2. Eerst testte ik andere hyperparameters, omdat dit een moeilijker multi-class probleem is.
3. Daarna probeerde ik extra features zoals Gr Liv Area en House Style.
4. Uiteindelijk bleek dat bij dit model vooral de hyperparameters de grootste winst gaven.

### Eindconclusie
- Voor Garage was de beste run: Experiment 3
- Voor Overall Qual was de beste run: Experiment 1

De nieuwe configuraties verbeterden dus niet altijd op accuracy, maar wél op macro F1-score.
Dat is hier belangrijker, omdat deze metriek eerlijker kijkt naar alle klassen.
